# RiverScope / PlanetScope: exploración y control de calidad

Este notebook inspecciona el conjunto público RiverScope antes de utilizarlo para segmentación de agua y análisis de costas. Conserva las etiquetas originales `0/1/2` y deriva una máscara binaria únicamente para visualización y estadísticas. No entrena modelos ni modifica el dataset.

Se revisan bandas y escalas, CRS y nodata, alineación imagen-máscara, calidad visual de los bordes, artefactos, balance de clases, duplicados y separación geográfica de las particiones.

In [ ]:
from pathlib import Path
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import BoundaryNorm, ListedColormap

from reservoir_segmentation.boundaries import (
    distance_squared_to_boundary,
    find_boundary_pixels,
    weights_from_distance_squared,
)
from reservoir_segmentation.dataset import (
    RIVERSCOPE_ARCHIVE_MD5,
    RIVERSCOPE_ARCHIVE_URL,
    RIVERSCOPE_RECORD_URL,
    download_riverscope_archive,
    extract_segmentation_archive,
    materialize_riverscope_sample,
    read_riverscope_split,
)

SEED = 2026
SPLIT = "train"
SAMPLE_COUNT = 6
DOWNLOAD_DATASET = False  # True descarga aproximadamente 8.1 GB de forma explícita.
PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = Path(os.environ.get("RIVERSCOPE_DATA", PROJECT_ROOT / "data" / "raw" / "riverscope"))
ARCHIVE_PATH = PROJECT_ROOT / "data" / "raw" / "RiverScope.zip"
EXTRACTED_ROOT = PROJECT_ROOT / "data" / "raw" / "riverscope"
SAMPLE_ROOT = PROJECT_ROOT / "data" / "processed" / "riverscope_sample"
random.seed(SEED)
np.random.seed(SEED)
print("Fuente oficial:", RIVERSCOPE_RECORD_URL)

## Adquisición reproducible

La descarga oficial es el registro de Zenodo de RiverScope. El ZIP contiene los CSV y los productos co-registrados; para este análisis se utilizan las rutas `normalized_planetscope_path` y `label_path` de PlanetScope. La descarga no se ejecuta automáticamente.

Para obtener la copia más fiel al dataset publicado, usar el ZIP oficial completo y no recomprimir ni convertir los GeoTIFF antes de inspeccionarlos. La función valida el MD5 publicado.

In [ ]:
if DOWNLOAD_DATASET:
    download_riverscope_archive(ARCHIVE_PATH)
    extract_segmentation_archive(ARCHIVE_PATH, EXTRACTED_ROOT)
    DATA_SOURCE = EXTRACTED_ROOT
elif ARCHIVE_PATH.is_file():
    DATA_SOURCE = ARCHIVE_PATH
elif DATA_ROOT.is_dir():
    DATA_SOURCE = DATA_ROOT
else:
    DATA_SOURCE = None

if DATA_SOURCE is None:
    print("No hay datos locales. Descarga:", RIVERSCOPE_ARCHIVE_URL)
    print("MD5 esperado:", RIVERSCOPE_ARCHIVE_MD5)
    print("Guardá el ZIP en data/raw/RiverScope.zip o extraelo en data/raw/riverscope.")
else:
    print("Fuente local:", DATA_SOURCE)

## CSV oficiales y muestra reproducible

Los CSV son la fuente de verdad para emparejar cada tile con su etiqueta; no se emparejan archivos por nombre. La muestra conserva `reach_id`, coordenadas, `planetscope_id` y las rutas originales. Solo se extraen los pares elegidos cuando la fuente es un ZIP.

In [ ]:
if DATA_SOURCE is None:
    split_tables, sample_rows = {}, []
else:
    split_tables = {split: pd.DataFrame(read_riverscope_split(DATA_SOURCE, split)) for split in ("train", "valid", "test")}
    sample_rows = materialize_riverscope_sample(DATA_SOURCE, SAMPLE_ROOT, split=SPLIT, count=SAMPLE_COUNT, seed=SEED)
    sample = pd.DataFrame(sample_rows)
    display(sample[["reach_id", "mid_lat", "mid_lon", "planetscope_id", "image_path", "mask_path"]])
    display(pd.DataFrame({"split": list(split_tables), "tiles": [len(table) for table in split_tables.values()]}))

## Inventario técnico y correspondencia espacial

RiverScope publica cuatro bandas PlanetScope en orden B, G, R, NIR, tiles de 500×500 y máscaras de un canal con valores 0, 1 y 2. No se convierten los valores radiométricos ni se reemplaza nodata por tierra.

In [ ]:
def read_pair(row):
    with rasterio.open(row["image_path"]) as image_src:
        image = image_src.read()
        image_meta = {"bands": image_src.count, "shape": (image_src.height, image_src.width), "dtype": str(image.dtype), "crs": str(image_src.crs), "transform": image_src.transform, "resolution": image_src.res, "nodata": image_src.nodata, "bounds": tuple(image_src.bounds)}
    with rasterio.open(row["mask_path"]) as mask_src:
        mask = mask_src.read(1)
        mask_meta = {"bands": mask_src.count, "shape": (mask_src.height, mask_src.width), "dtype": str(mask.dtype), "crs": str(mask_src.crs), "transform": mask_src.transform, "resolution": mask_src.res, "nodata": mask_src.nodata, "bounds": tuple(mask_src.bounds)}
    return image, mask, image_meta, mask_meta

inventory = []
for row in sample_rows:
    image, mask, image_meta, mask_meta = read_pair(row)
    values, counts = np.unique(mask, return_counts=True)
    same_grid = image.shape[1:] == mask.shape and image_meta["crs"] == mask_meta["crs"] and image_meta["transform"] == mask_meta["transform"]
    inventory.append({"id": row["planetscope_id"], "shape": image.shape, "mask_shape": mask.shape, "image_dtype": image_meta["dtype"], "mask_dtype": mask_meta["dtype"], "crs": image_meta["crs"], "resolution": image_meta["resolution"], "image_nodata": image_meta["nodata"], "mask_nodata": mask_meta["nodata"], "aligned": same_grid, "label_values": dict(zip(values.astype(int), counts.astype(int)))})
inventory_df = pd.DataFrame(inventory)
display(inventory_df)
if inventory:
    observed = sorted(set().union(*(set(item["label_values"]) for item in inventory)))
    print("Valores observados:", observed, "| esperados: [0, 1, 2]")

## Galería, zoom de costa y composiciones

La máscara original se muestra sin alterar. La máscara binaria considera `1` y `2` como agua. Las composiciones RGB y NIR/R/G se estiran por percentiles solo para facilitar la inspección. Revisar desplazamientos, nubes, sombras, artefactos y bordes del tile que no representen costas físicas.

In [ ]:
LABEL_CMAP = ListedColormap([(0.18, 0.19, 0.21), (0.08, 0.68, 0.91), (0.84, 0.31, 0.94)])
LABEL_NORM = BoundaryNorm(np.arange(-0.5, 3.5), LABEL_CMAP.N)

def composite(image, order):
    channels = image[list(order)].astype(np.float32)
    result = np.zeros((*channels.shape[1:], 3), dtype=np.float32)
    for output_channel, channel in enumerate(channels):
        finite = channel[np.isfinite(channel)]
        if finite.size:
            low, high = np.percentile(finite, [2, 98])
            result[..., output_channel] = np.clip((channel - low) / max(high - low, 1e-12), 0, 1)
    return result

def plot_gallery(rows, max_rows=4):
    rows = rows[:max_rows]
    if not rows:
        print("No hay muestras para mostrar.")
        return
    fig, axes = plt.subplots(len(rows), 5, figsize=(18, 4 * len(rows)), squeeze=False, constrained_layout=True)
    for row_index, row in enumerate(rows):
        image, mask, *_ = read_pair(row)
        binary = np.isin(mask, [1, 2])
        rgb, false_color = composite(image, (2, 1, 0)), composite(image, (3, 1, 0))
        axes[row_index, 0].imshow(rgb); axes[row_index, 0].set_title(f"{row['planetscope_id']} · RGB")
        axes[row_index, 1].imshow(false_color); axes[row_index, 1].set_title("NIR / G / B")
        axes[row_index, 2].imshow(mask, cmap=LABEL_CMAP, norm=LABEL_NORM, interpolation="nearest"); axes[row_index, 2].set_title("Etiqueta original 0/1/2")
        axes[row_index, 3].imshow(rgb); axes[row_index, 3].imshow(np.ma.masked_where(~binary, binary), cmap=ListedColormap(["cyan"]), alpha=0.38); axes[row_index, 3].set_title("Agua binaria")
        axes[row_index, 4].imshow(rgb)
        if binary.any() and (~binary).any(): axes[row_index, 4].contour(binary.astype(float), levels=[0.5], colors="yellow", linewidths=0.8)
        axes[row_index, 4].set_title("Contorno sobre RGB")
        for axis in axes[row_index]: axis.set_axis_off()
    plt.show()

plot_gallery(sample_rows)

if sample_rows:
    image, mask, *_ = read_pair(sample_rows[0])
    binary = np.isin(mask, [1, 2])
    boundary = find_boundary_pixels(binary)
    points = np.argwhere(boundary)
    y, x = points[len(points) // 2] if len(points) else (mask.shape[0] // 2, mask.shape[1] // 2)
    side = min(180, *mask.shape)
    x0, y0 = max(0, min(int(x) - side // 2, mask.shape[1] - side)), max(0, min(int(y) - side // 2, mask.shape[0] - side))
    region = np.s_[y0:y0 + side, x0:x0 + side]
    rgb = composite(image, (2, 1, 0))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    axes[0].imshow(rgb[region]); axes[0].set_title("Zoom RGB")
    axes[1].imshow(rgb[region]); axes[1].contour(binary[region].astype(float), levels=[0.5], colors="yellow"); axes[1].set_title("Zoom con contorno")
    axes[2].imshow(distance_squared_to_boundary(boundary)[region], cmap="magma"); axes[2].set_title("Distancia² a la costa")
    for axis in axes: axis.set_axis_off()
    plt.show()

## Distancias y mapas de pesos

`d²` depende solo de la máscara original. Los mapas derivados usan `w = 1 + alpha · exp(-d² / (2 · sigma²))`; cambiar sigma o alpha no debe recalcular la transformada geométrica.

In [ ]:
SIGMAS, ALPHAS = (1.5, 3.0, 6.0, 12.0), (1.0, 4.0, 10.0)
if sample_rows:
    d2 = distance_squared_to_boundary(find_boundary_pixels(np.isin(read_pair(sample_rows[0])[1], [1, 2])))
    fig, axes = plt.subplots(len(ALPHAS), len(SIGMAS), figsize=(16, 10), constrained_layout=True)
    for row_index, alpha in enumerate(ALPHAS):
        for column_index, sigma in enumerate(SIGMAS):
            axes[row_index, column_index].imshow(weights_from_distance_squared(d2, sigma, peak_weight=alpha), cmap="viridis")
            axes[row_index, column_index].set_title(f"sigma={sigma}, alpha={alpha}"); axes[row_index, column_index].set_axis_off()
    plt.show()
    print("El mapa d² se calculó una sola vez para esta máscara.")

## Balance de clases y particiones

Se buscan IDs repetidos, `reach_id` compartidos, coordenadas centrales repetidas y etiquetas inesperadas. El test no se usa para elegir hiperparámetros.

In [ ]:
if split_tables:
    all_rows = pd.concat([table.assign(split=split) for split, table in split_tables.items()], ignore_index=True)
    display(all_rows.groupby("split").size().rename("tiles").to_frame())
    display(all_rows.groupby("reach_id")["split"].nunique().loc[lambda values: values > 1].to_frame("particiones"))
    display(all_rows[all_rows.duplicated("planetscope_id", keep=False)][["split", "planetscope_id", "reach_id"]].sort_values("planetscope_id"))
    display(all_rows[all_rows.duplicated(["mid_lat", "mid_lon"], keep=False)][["split", "planetscope_id", "mid_lat", "mid_lon"]])
else:
    print("Cargá el dataset para auditar las particiones.")

## Registro de casos favorables y problemáticos

Completar una fila por tile después de revisar los gráficos. La puntuación es una valoración visual de cuánto sigue la etiqueta a la costa observable; no es una métrica de exactitud.

In [ ]:
review_columns = ["planetscope_id", "reach_id", "split", "score_1_to_5", "notes"]
review = pd.DataFrame([{"planetscope_id": row["planetscope_id"], "reach_id": row["reach_id"], "split": SPLIT, "score_1_to_5": "", "notes": ""} for row in sample_rows], columns=review_columns)
display(review)
print("Guardar la tabla completada en data/processed/riverscope_quality_review.csv")

### Criterio de uso

Antes de entrenar, las imágenes y etiquetas deben pasar una revisión visual de alineación, nodata, artefactos y costas. Las observaciones problemáticas se conservan junto al identificador del tile; no se corrigen silenciosamente las etiquetas ni se tratan píxeles no confiables como tierra.